# VGAE vs R-GCN Latent Smoothness

This notebook compares the Wikidata latent spaces with a category-aware workflow:

- Use a semantic relation such as `occupation` to assign human nodes to broader categories.
- Run separate PCA fits for the R-GCN and VGAE latent spaces.
- Show the two PCA views side by side instead of forcing both models into one shared basis.
- Run KMeans directly in each latent space and compare how well the learned clusters line up with the semantic categories.

The default setup uses human nodes grouped into broad occupation families such as `military`, `law_judiciary`, `religious`, and `entertainment_media`, which is much more interpretable than the old shared projection over exact occupation labels.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'R-GCN' / 'rgcn_model.py').exists():
            return candidate
    raise RuntimeError('Could not find project root containing R-GCN/rgcn_model.py')


PROJECT_ROOT = find_project_root(Path.cwd())
RGCN_DIR = PROJECT_ROOT / 'R-GCN'
if str(RGCN_DIR) not in sys.path:
    sys.path.insert(0, str(RGCN_DIR))

from latent_space_analysis import (
    DEVICE,
    add_kmeans_clusters,
    build_semantic_category_frame,
    compare_model_metrics,
    load_latent_artifacts,
    make_category_color_map,
    plot_side_by_side,
    project_latents_with_pca,
)

artifacts = load_latent_artifacts(PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')
print(f'Graph checkpoint: {artifacts.graph_checkpoint}')
print(f'R-GCN checkpoint: {artifacts.rgcn_checkpoint}')
print(f'VGAE results: {artifacts.vgae_results_path}')
print(f'R-GCN latents: {artifacts.rgcn_latents.shape}')
print(f'VGAE latents: {artifacts.vgae_mu.shape}')
print(f'VGAE posterior std mean: {artifacts.vgae_std.mean():.4f}')
print(f'Human node cache size: {len(artifacts.human_node_ids):,}')

In [ ]:
CATEGORY_RELATION = 'occupation'
COLLAPSE_OCCUPATION_FAMILIES = True
TOP_K_CATEGORIES = 8
MIN_CATEGORY_SIZE = 10
MAX_NODES_PER_CATEGORY = 250
RANDOM_SEED = 42

print('Semantic grouping relation:', CATEGORY_RELATION)
print('Collapse occupation labels into broad families:', COLLAPSE_OCCUPATION_FAMILIES)
print('Top categories kept:', TOP_K_CATEGORIES)
print('Minimum nodes per category:', MIN_CATEGORY_SIZE)
print('Per-category sample cap:', MAX_NODES_PER_CATEGORY)

## Human Categories

Each human node is assigned to the strongest high-support category induced by the chosen relation. With the default settings, raw occupation labels are collapsed into broader families before ranking and sampling.

In [ ]:
category_frame, category_summary = build_semantic_category_frame(
    graph=artifacts.graph,
    subject_node_ids=artifacts.human_node_ids,
    relation_label=CATEGORY_RELATION,
    label_to_rel=artifacts.label_to_rel,
    top_k=TOP_K_CATEGORIES,
    min_category_size=MIN_CATEGORY_SIZE,
    max_nodes_per_category=MAX_NODES_PER_CATEGORY,
    seed=RANDOM_SEED,
    collapse_occupation_families=COLLAPSE_OCCUPATION_FAMILIES,
)
category_frame['qid'] = category_frame['node_id'].map(artifacts.node_to_qid)

display(category_summary)
display(
    category_frame[['category', 'raw_category', 'label', 'qid', 'node_id']]
    .groupby('category', group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

print(f'Sampled nodes kept for visualization and metrics: {len(category_frame):,}')

## Separate PCA Projections

This is the key fix: R-GCN and VGAE each get their own PCA fit. That way we compare their structure, not an artifact of forcing both latent spaces into a single shared projection.

In [ ]:
rgcn_projection, rgcn_pca = project_latents_with_pca(
    artifacts.rgcn_latents,
    category_frame,
    model_name='R-GCN',
)
vgae_projection, vgae_pca = project_latents_with_pca(
    artifacts.vgae_mu,
    category_frame,
    model_name='VGAE',
)

rgcn_clustered = add_kmeans_clusters(rgcn_projection, artifacts.rgcn_latents)
vgae_clustered = add_kmeans_clusters(vgae_projection, artifacts.vgae_mu)

category_color_map = make_category_color_map(category_summary['category'].astype(str).tolist())

print(f'R-GCN PCA variance explained: {rgcn_pca.explained_variance_ratio_.sum():.2%}')
print(f'VGAE PCA variance explained: {vgae_pca.explained_variance_ratio_.sum():.2%}')

In [ ]:
fig = plot_side_by_side(
    rgcn_projection,
    vgae_projection,
    color_col='category',
    title='Human nodes colored by semantic category in separate PCA views',
    left_title=f'R-GCN ({rgcn_pca.explained_variance_ratio_.sum():.1%} variance)',
    right_title=f'VGAE ({vgae_pca.explained_variance_ratio_.sum():.1%} variance)',
    color_map=category_color_map,
)
display(fig)

## Smoothness And Cluster Alignment

The table below compares the two latent spaces using:

- `knn_label_agreement`: how often nearby points share the same semantic category.
- `silhouette_by_category`: how well those semantic categories separate in latent space.
- `cluster_nmi`, `cluster_ari`, `cluster_purity`: how well unsupervised KMeans clusters match the semantic grouping.

In [ ]:
metrics = compare_model_metrics(
    category_frame,
    artifacts.rgcn_latents,
    artifacts.vgae_mu,
)
display(metrics)

for model_name, clustered in [('R-GCN', rgcn_clustered), ('VGAE', vgae_clustered)]:
    print(f'\n{model_name} category x cluster contingency')
    display(pd.crosstab(clustered['category'], clustered['cluster']))

In [ ]:
cluster_names = sorted(
    set(rgcn_clustered['cluster'].astype(str)).union(vgae_clustered['cluster'].astype(str))
)
cluster_color_map = make_category_color_map(cluster_names)

cluster_fig = plot_side_by_side(
    rgcn_clustered,
    vgae_clustered,
    color_col='cluster',
    title='Same PCA views colored by KMeans clusters learned in each latent space',
    left_title='R-GCN KMeans clusters',
    right_title='VGAE KMeans clusters',
    color_map=cluster_color_map,
)
display(cluster_fig)

## Notes

- To try a different semantic grouping, change `CATEGORY_RELATION` to another relation label present in `artifacts.label_to_rel`.
- If you want denser or sparser plots, adjust `MAX_NODES_PER_CATEGORY`.
- The cleanest comparison is usually the semantic-category PCA view plus the metric table; the cluster-colored plot is mainly there to show what KMeans is finding without supervision.